# Training Comparison Analysis

This notebook helps compare training runs between baseline and optimized configurations.

Compare:
- Training speed (time per epoch)
- Loss curves
- Metric improvements
- GPU utilization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## Load Training Data

In [ ]:
# Baseline (original fsdp_checkpoints9)
baseline_csv = "../../crystalpred/fsdp_checkpoints9/training_metrics.csv"

# New runs
optimized_csv = "../checkpoints/training_metrics.csv"

# Load data
baseline = None
if Path(baseline_csv).exists():
    baseline = pd.read_csv(baseline_csv)
    print(f"✅ Loaded baseline: {len(baseline)} epochs")
else:
    print(f"⚠️  Baseline not found: {baseline_csv}")

optimized = None
if Path(optimized_csv).exists():
    optimized = pd.read_csv(optimized_csv)
    print(f"✅ Loaded optimized: {len(optimized)} epochs")
else:
    print(f"⚠️  Optimized not found: {optimized_csv}")

## Training Loss Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Validation Loss
ax = axes[0]
if baseline is not None:
    ax.plot(baseline['epoch'], baseline['val_loss'], 
            label='Baseline (min loss)', linewidth=2, alpha=0.8)
if optimized is not None:
    ax.plot(optimized['epoch'], optimized['val_loss'], 
            label='Optimized (weighted loss)', linewidth=2, alpha=0.8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title('Validation Loss Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Training Loss
ax = axes[1]
if baseline is not None:
    ax.plot(baseline['epoch'], baseline['train_loss'], 
            label='Baseline', linewidth=2, alpha=0.8)
if optimized is not None:
    ax.plot(optimized['epoch'], optimized['train_loss'], 
            label='Optimized', linewidth=2, alpha=0.8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print best values
if baseline is not None:
    print(f"\n📊 Baseline Best Val Loss: {baseline['val_loss'].min():.4f} at epoch {baseline['val_loss'].idxmin() + 1}")
if optimized is not None:
    print(f"📊 Optimized Best Val Loss: {optimized['val_loss'].min():.4f} at epoch {optimized['val_loss'].idxmin() + 1}")

## Precision Metrics Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics = [
    ('val_precision_L', 'Precision@L'),
    ('val_precision_L2', 'Precision@L/2'),
    ('val_precision_L5', 'Precision@L/5'),
    ('val_recall', 'Recall')
]

for idx, (metric, title) in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    if baseline is not None and metric in baseline.columns:
        ax.plot(baseline['epoch'], baseline[metric], 
                label='Baseline', linewidth=2, alpha=0.8, marker='o', markersize=3)
    if optimized is not None and metric in optimized.columns:
        ax.plot(optimized['epoch'], optimized[metric], 
                label='Optimized', linewidth=2, alpha=0.8, marker='s', markersize=3)
    
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'{title} Over Time', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
print("\n" + "="*60)
print("FINAL METRICS COMPARISON")
print("="*60)

if baseline is not None:
    last_epoch = baseline.iloc[-1]
    print(f"\n📊 Baseline (Epoch {int(last_epoch['epoch'])}):")
    print(f"  Val Loss:        {last_epoch['val_loss']:.4f}")
    print(f"  Precision@L:     {last_epoch['val_precision_L']:.4f}")
    print(f"  Precision@L/2:   {last_epoch['val_precision_L2']:.4f}")
    print(f"  Precision@L/5:   {last_epoch['val_precision_L5']:.4f}")
    print(f"  Recall:          {last_epoch['val_recall']:.4f}")

if optimized is not None:
    last_epoch = optimized.iloc[-1]
    print(f"\n📊 Optimized (Epoch {int(last_epoch['epoch'])}):")
    print(f"  Val Loss:        {last_epoch['val_loss']:.4f}")
    print(f"  Precision@L:     {last_epoch['val_precision_L']:.4f}")
    print(f"  Precision@L/2:   {last_epoch['val_precision_L2']:.4f}")
    print(f"  Precision@L/5:   {last_epoch['val_precision_L5']:.4f}")
    print(f"  Recall:          {last_epoch['val_recall']:.4f}")

if baseline is not None and optimized is not None:
    print(f"\n📈 Improvement:")
    baseline_last = baseline.iloc[-1]
    optimized_last = optimized.iloc[-1]
    
    loss_diff = optimized_last['val_loss'] - baseline_last['val_loss']
    prec_diff = optimized_last['val_precision_L'] - baseline_last['val_precision_L']
    
    print(f"  Val Loss:        {loss_diff:+.4f} ({'better' if loss_diff < 0 else 'worse'})")
    print(f"  Precision@L:     {prec_diff:+.4f} ({'better' if prec_diff > 0 else 'worse'})")

## Learning Rate Analysis

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

if baseline is not None:
    ax.plot(baseline['epoch'], baseline['lr'], 
            label='Baseline (plateau scheduler)', linewidth=2, alpha=0.8)
if optimized is not None:
    ax.plot(optimized['epoch'], optimized['lr'], 
            label='Optimized (cosine scheduler)', linewidth=2, alpha=0.8)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Plateau Detection

In [ ]:
def detect_plateau(series, window=5, threshold=0.001):
    """Detect when loss stopped improving significantly."""
    rolling_improvement = -series.diff().rolling(window).mean()
    plateau_start = None
    
    for idx in range(window, len(rolling_improvement)):
        if rolling_improvement.iloc[idx] < threshold:
            if plateau_start is None:
                plateau_start = idx - window
    
    return plateau_start

if baseline is not None:
    plateau = detect_plateau(baseline['val_loss'])
    if plateau:
        print(f"📉 Baseline plateaued around epoch {plateau}")
        print(f"   Best loss before plateau: {baseline['val_loss'].iloc[:plateau].min():.4f}")
        print(f"   Final loss: {baseline['val_loss'].iloc[-1]:.4f}")
        print(f"   Improvement after plateau: {baseline['val_loss'].iloc[plateau] - baseline['val_loss'].iloc[-1]:.4f}")

if optimized is not None and len(optimized) > 10:
    plateau = detect_plateau(optimized['val_loss'])
    if plateau:
        print(f"\n📉 Optimized plateaued around epoch {plateau}")
        print(f"   Best loss before plateau: {optimized['val_loss'].iloc[:plateau].min():.4f}")
        print(f"   Final loss: {optimized['val_loss'].iloc[-1]:.4f}")
    else:
        print(f"\n✅ Optimized: No plateau detected yet! Still improving.")

## Summary Statistics

In [ ]:
import pandas as pd

summary_data = []

if baseline is not None:
    summary_data.append({
        'Config': 'Baseline',
        'Best Val Loss': baseline['val_loss'].min(),
        'Final Val Loss': baseline['val_loss'].iloc[-1],
        'Best Precision@L': baseline['val_precision_L'].max(),
        'Final Precision@L': baseline['val_precision_L'].iloc[-1],
        'Epochs': len(baseline),
        'Initial LR': baseline['lr'].iloc[0],
        'Final LR': baseline['lr'].iloc[-1]
    })

if optimized is not None:
    summary_data.append({
        'Config': 'Optimized',
        'Best Val Loss': optimized['val_loss'].min(),
        'Final Val Loss': optimized['val_loss'].iloc[-1],
        'Best Precision@L': optimized['val_precision_L'].max(),
        'Final Precision@L': optimized['val_precision_L'].iloc[-1],
        'Epochs': len(optimized),
        'Initial LR': optimized['lr'].iloc[0],
        'Final LR': optimized['lr'].iloc[-1]
    })

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print("\n" + "="*80)
    print("TRAINING SUMMARY")
    print("="*80)
    print(summary_df.to_string(index=False))

## Performance Metrics (If Available)

Run training with time tracking to analyze speed improvements.

In [ ]:
# If you track time per epoch, analyze it here
# Example:
# baseline_time = 3.0  # hours per epoch
# optimized_time = 2.1  # hours per epoch (estimated with compile)
# speedup = baseline_time / optimized_time
# print(f"⚡ Speedup: {speedup:.2f}x faster ({baseline_time:.1f}h → {optimized_time:.1f}h per epoch)")